# RAG Pipeline: Data Ingestion - Vector DB

In [122]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from pathlib import Path

In [123]:
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_directory_path = Path(pdf_directory)
    pdf_files = list(pdf_directory_path.glob("*.pdf"))
    
    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}...")
        # Load the PDF using PyPDFLoader
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            for doc in documents:
                doc.metadata["source"] = str(pdf_file)
                doc.metadata["file_type"] = "pdf"
            all_documents.extend(documents)
            print(f"Successfully processed {len(documents)}.")
        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")
    return all_documents

# Text Splitting get into chunks

In [124]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")
    if split_docs:
        print(f"First chunk preview: {split_docs[0].page_content[:200]}...")
        print(f"First chunk metadata: {split_docs[0].metadata}")
    return split_docs

In [125]:
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
import numpy as np
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

# Creating Embedding Model

In [126]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Loaded model: {self.model_name} and dimensions: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model is not loaded.")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings for {len(texts)} texts with shape: {embeddings.shape}")
        return embeddings
    

# Creating VectoreStore

In [127]:
class VectoreStore:
    def __init__(self, collection_name: str = "pdfdocuments", persist_directory: str = "../data/vectore_storedb"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok = True)
            self.client = chromadb.PersistentClient(path = self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name, 
                metadata={
                    "description": "Collection of PDF document embeddings",
                    "hnsw:space": "cosine"
                }
            )
            print(f"Initialized vector store with collection: {self.collection_name}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Document], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        print(f"Adding {len(documents)} documents to the vector store...")

        # prepare data for chromadb
        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []
        for i, (doc, emb) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadatas.append(doc.metadata)
            documents_texts.append(doc.page_content)
            embeddings_list.append(emb.tolist())
        try:
            self.collection.add(
                ids = ids,
                metadatas = metadatas,
                documents = documents_texts,
                embeddings = embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise



In [128]:
all_pdfs = process_all_pdfs("../data/pdfs") #creating LangChain Document objects from all PDFs in the specified directory
chunks = split_documents(all_pdfs) # chunking the documents into smaller pieces for better embedding and retrieval

embedding_manager = EmbeddingManager()
texts = [doc.page_content for doc in chunks] #creating a list of text content from the document chunks for embedding generation
embeddings = embedding_manager.generate_embeddings(texts) #generating embeddings for the text content of the document chunks

vector_store = VectoreStore()
vector_store.add_documents(chunks, embeddings) #vector store is populated with the document chunks and their corresponding embeddings for efficient retrieval and similarity search

Processing ../data/pdfs/03_it_security_and_data_handling.pdf...
Successfully processed 4.
Processing ../data/pdfs/05_products_customers_and_support_playbook.pdf...
Successfully processed 4.
Processing ../data/pdfs/02_people_and_workplace_policy.pdf...
Successfully processed 4.
Processing ../data/pdfs/01_company_overview.pdf...
Successfully processed 3.
Processing ../data/pdfs/04_travel_expense_and_procurement.pdf...
Successfully processed 4.
Split 19 documents into 48 chunks.
First chunk preview: ASTERION WORKS  |  FICTIONAL RAG TRAINING DATA
Page 1
ASTERION WORKS
 IT Security, Access & Data Handling
 Standard
 Identity, devices, customer data, and incident response
Document ID
03_IT_SECURITY_...
First chunk metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-08T10:05:15+00:00', 'author': 'Asterion Works', 'keywords': '', 'moddate': '2026-11-08T10:05:15+00:00', 'subject': '(unspecified)', 'title': 'IT Security, Access & Da

Batches: 100%|██████████| 2/2 [00:00<00:00, 10.50it/s]

Generated embeddings for 48 texts with shape: (48, 384)


Initialized vector store with collection: pdfdocuments
Adding 48 documents to the vector store...
Successfully added 48 documents to the vector store.


In [129]:
all_pdfs = process_all_pdfs("../data/pdfs") #creating LangChain Document objects from all PDFs in the specified directory
chunks = split_documents(all_pdfs) # chunking the documents into smaller pieces for better embedding and retrieval

embedding_manager = EmbeddingManager()
texts = [doc.page_content for doc in chunks] #creating a list of text content from the document chunks for embedding generation
embeddings = embedding_manager.generate_embeddings(texts) #generating embeddings for the text content of the document chunks

vector_store = VectoreStore()
vector_store.add_documents(chunks, embeddings) #vector store is populated with the document chunks and their corresponding embeddings for efficient retrieval and similarity search

Processing ../data/pdfs/03_it_security_and_data_handling.pdf...
Successfully processed 4.
Processing ../data/pdfs/05_products_customers_and_support_playbook.pdf...
Successfully processed 4.
Processing ../data/pdfs/02_people_and_workplace_policy.pdf...
Successfully processed 4.
Processing ../data/pdfs/01_company_overview.pdf...
Successfully processed 3.
Processing ../data/pdfs/04_travel_expense_and_procurement.pdf...
Successfully processed 4.
Split 19 documents into 48 chunks.
First chunk preview: ASTERION WORKS  |  FICTIONAL RAG TRAINING DATA
Page 1
ASTERION WORKS
 IT Security, Access & Data Handling
 Standard
 Identity, devices, customer data, and incident response
Document ID
03_IT_SECURITY_...
First chunk metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-08T10:05:15+00:00', 'author': 'Asterion Works', 'keywords': '', 'moddate': '2026-11-08T10:05:15+00:00', 'subject': '(unspecified)', 'title': 'IT Security, Access & Da

Batches: 100%|██████████| 2/2 [00:00<00:00, 10.16it/s]

Generated embeddings for 48 texts with shape: (48, 384)


Initialized vector store with collection: pdfdocuments
Adding 48 documents to the vector store...
Successfully added 48 documents to the vector store.


# RAG Retriever pipeline

In [ ]:
class RAGRetriever:
    def __init__(self, vectore_store: VectoreStore, embedding_manager: EmbeddingManager):
        self.vectore_store = vectore_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        # Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Retrieve all embeddings and metadata from the vector store
        try:
            results = self.vectore_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results = top_k,
            )
        except Exception as e:
            print(f"Error querying vector store: {e}")
            return []

        retrieved_docs = []
        if results['documents'] and results['documents'][0]:
            documents = results['documents'][0]
            metadatas = results['metadatas'][0]
            distances = results['distances'][0]
            ids = results['ids'][0]
            print(f"Retrieved {len(documents)} documents from vector store.")
            for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                similarity_score = 1 - distance  # Convert distance to similarity score
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "content": document,
                        "metadata": metadata,
                        "similarity_score": similarity_score,
                        "distance": distance,
                        "rank": i + 1
                    })
                else:
                    print(f"Document {doc_id} filtered out due to low similarity score: {similarity_score}")
        else:
            print("No documents retrieved from vector store.")
        
        return retrieved_docs
        

In [131]:
rag_retriever = RAGRetriever(vector_store, embedding_manager)
docs = rag_retriever.retrieve("What are the company's rules for employees accessing confidential customer data remotely?", top_k=5, score_threshold=0.0)
print(f"Retrieved {len(docs)} documents: {docs}")

Batches: 100%|██████████| 1/1 [00:00<00:00,  6.54it/s]

Generated embeddings for 1 texts with shape: (1, 384)
1 {'ids': [['doc_d6f5b360_3', 'doc_1bb50fab_3', 'doc_e730fac8_4', 'doc_c02b209b_4', 'doc_cbd0a53f_2']], 'embeddings': None, 'documents': [['\x7f Personal devices may access webmail only through the managed browser profile. Downloading customer data to\na personal device is prohibited.\n\x7f USB storage is blocked by default. A time-limited exception requires approval from IT Security and the data owner.\n\x7f Lost or stolen devices must be reported to security@asterion-works.example within 30 minutes of discovery, even\nif the device is believed to be encrypted.\n4. Data classification\nPublic: Approved for unrestricted public distribution.\nInternal: Routine company information. Share only with employees and approved contractors who need it.\nConfidential: Business-sensitive information such as contracts, employee records, unreleased financials, and\nnon-public product plans. Share only with named, authorised recipients.\nRestricte

# Integration Vectordb context pipeline with LLM output

In [132]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(api_key=api_key, model="openai/gpt-oss-120b", temperature=0.1, max_tokens=1024)

def rag_simple_query(query: str, top_k: int = 3) -> str:
    # Retrieve relevant documents using RAGRetriever
    retrieved_docs = rag_retriever.retrieve(query, top_k=top_k)
    
    if not retrieved_docs:
        return "No relevant documents found."
    
    # Concatenate the content of the retrieved documents
    context = "\n\n".join([doc["content"] for doc in retrieved_docs])
    
    # Create a prompt for the LLM
    prompt = f"""Use the following context to answer the question concisely.
                Context: {context}
                Question: {query}
                Answer:"""
    
    # Generate response using the LLM
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    return response.content

In [135]:
answer = rag_simple_query("What are the company's rules for employees accessing confidential customer data remotely?", top_k=3)
print(f"Answer: {answer}")

Batches: 100%|██████████| 1/1 [00:00<00:00, 16.42it/s]


Generated embeddings for 1 texts with shape: (1, 384)
1 {'ids': [['doc_d6f5b360_3', 'doc_1bb50fab_3', 'doc_e730fac8_4']], 'embeddings': None, 'documents': [['\x7f Personal devices may access webmail only through the managed browser profile. Downloading customer data to\na personal device is prohibited.\n\x7f USB storage is blocked by default. A time-limited exception requires approval from IT Security and the data owner.\n\x7f Lost or stolen devices must be reported to security@asterion-works.example within 30 minutes of discovery, even\nif the device is believed to be encrypted.\n4. Data classification\nPublic: Approved for unrestricted public distribution.\nInternal: Routine company information. Share only with employees and approved contractors who need it.\nConfidential: Business-sensitive information such as contracts, employee records, unreleased financials, and\nnon-public product plans. Share only with named, authorised recipients.\nRestricted: Customer personal data, authentic